<a href="https://colab.research.google.com/github/rfinean/pydata-workshop-2023/blob/main/Dufours_BMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load and Append Data from Google Sheets

To interact with Google Sheets directly, we'll use `gspread`. First, we need to install the necessary libraries.

In [4]:
!pip install gspread gspread-dataframe

Next, we need to authenticate `gspread` to access your Google Sheets. This will open a new window in your browser to complete the authentication process.

In [5]:
import gspread
from google.colab import auth
from gspread_dataframe import get_as_dataframe
import google.auth

auth.authenticate_user()
creds, project = google.auth.default() # Get the user's credentials
gc = gspread.authorize(creds) # Authorize gspread with these credentials
print('Authentication successful.')

Authentication successful.


Now, let's load the first spreadsheet into a pandas DataFrame. We'll use the URL to open the spreadsheet and then read the specified worksheet (using `gid` to identify the worksheet).

In [6]:
import pandas as pd

# URL for the Main BMS Data spreadsheet
spreadsheet_url_1 = 'https://docs.google.com/spreadsheets/d/1q0zRnz7IYCFcBFjedQXHbz3mQYi7Jqjm2pSmKeiwOSg/edit'
worksheet_gid_1 = '2024167550' # gid for the specific 2026 raw data worksheet gid=2024167550

try:
    # Open the spreadsheet by URL
    sh1 = gc.open_by_url(spreadsheet_url_1)
    # Select the worksheet by GID (index is usually 0 for the first sheet if GID is not specified)
    # If the GID is needed, you might need to iterate through sh.worksheets() to find it or use a specific index
    # For this example, assuming the GID 2123714432 corresponds to a readable sheet
    worksheet1 = sh1.get_worksheet_by_id(worksheet_gid_1)

    # Read the data into a DataFrame
    df1 = get_as_dataframe(worksheet1, header=0)

    print(f"Successfully loaded data from the first spreadsheet (GID: {worksheet_gid_1}).")
    display(df1.head())
except Exception as e:
    print(f"Error loading BMS Data spreadsheet: {e}")
    print("Please ensure the spreadsheet is publicly accessible or you have given Colab access during authentication.")

Error loading BMS Data spreadsheet: id 2024167550 not found
Please ensure the spreadsheet is publicly accessible or you have given Colab access during authentication.


Next, we'll load the second spreadsheet into another pandas DataFrame, using its URL and GID.

In [7]:
# URL for the second spreadsheet
spreadsheet_url_2 = 'https://docs.google.com/spreadsheets/d/1wNgl1uPUQju6vLBoywy4KiMSvEfsHgJjH2D36krfauM/edit'
worksheet_gid_2 = '717402912' # gid for the specific worksheet

try:
    # Open the spreadsheet by URL
    sh2 = gc.open_by_url(spreadsheet_url_2)
    # Select the worksheet by GID
    worksheet2 = sh2.get_worksheet_by_id(worksheet_gid_2)

    # Read the data into a DataFrame
    df2 = get_as_dataframe(worksheet2, header=0)

    print(f"Successfully loaded data from the second spreadsheet (GID: {worksheet_gid_2}).")
    display(df2.head())
except Exception as e:
    print(f"Error loading second spreadsheet: {e}")
    print("Please ensure the spreadsheet is publicly accessible or you have given Colab access during authentication.")

Successfully loaded data from the second spreadsheet (GID: 717402912).


,Asset Name,System Id,Date,Time,Parameter,Value
0,Dufours Place BMS,80412362.0,11-06-2026,00:45,UPS_FLT,0.00 Off/On
1,Dufours Place BMS,80412362.0,11-06-2026,00:45,SUP_FAN_1_HND_POS,0
2,Dufours Place BMS,80412362.0,11-06-2026,00:45,SUP_FAN_FLW_STS,-1
3,Dufours Place BMS,80412362.0,11-06-2026,00:45,SUP_FAN_2_AUTO_POS,-1
4,Dufours Place BMS,80412362.0,11-06-2026,00:45,SUP_FAN_2_HND_POS,0


As requested, let's transform `df2` (which contains key-value pairs) into a new DataFrame `df3`. This DataFrame will have a datetime index derived from the 'Date' and 'Time' columns, and columns representing each unique 'Parameter' from `df2`.

In [8]:
import numpy as np

# Create a combined 'DateTime' column from 'Date' and 'Time'
# Assuming 'Date' is in DD-MM-YYYY format and 'Time' is HH:MM
# Handle potential errors during conversion by coercing invalid dates to NaT
df2['DateTime'] = pd.to_datetime(df2['Date'] + ' ' + df2['Time'], format='%d-%m-%Y %H:%M', errors='coerce')

# Drop rows where DateTime conversion failed
df2.dropna(subset=['DateTime'], inplace=True)

# Attempt to convert 'Value' to numeric, coercing errors to NaN
df2['Value'] = pd.to_numeric(df2['Value'], errors='coerce')

# Pivot the DataFrame to get parameters as columns and DateTime as index
df3 = df2.pivot_table(index='DateTime', columns='Parameter', values='Value', aggfunc='first')

# Ensure the index is unique by dropping duplicates, keeping the first occurrence
df3 = df3[~df3.index.duplicated(keep='first')]

# Negate all numerical values in df3 as requested
# We'll apply this only to columns that are numerical types
for col in df3.select_dtypes(include=np.number).columns:
    df3[col] = 0 - df3[col]

# Reset index to make 'DateTime' a column if preferred, or keep it as index
# df3 = df3.reset_index()

# Display the new DataFrame's head and info
print("DataFrame df3 created successfully (with negated values):")
display(df3.head())
print("\nInformation about df3:")
df3.info()

DataFrame df3 created successfully (with negated values):


Parameter,B1_AUTO_POS,B1_DIV_VLV_CLSD,B1_DIV_VLV_OPN,B1_EN,B1_GAS_BSTR_STS,B1_HND_POS,B1_HTCO,B1_LKOUT,B1_RUN,B2_AUTO_POS,...,SUP_FAN_1_EN,SUP_FAN_1_FLT,SUP_FAN_1_HND_POS,SUP_FAN_2_AUTO_POS,SUP_FAN_2_EN,SUP_FAN_2_FLT,SUP_FAN_2_HND_POS,SUP_FAN_FLW_STS,THRML_LNK_STS,WTR_LK_DETEC
DateTime,,,,,,,,,,,,,,,,,,,,,
2026-03-01 00:00:00,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2026-03-01 00:15:00,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2026-03-01 00:30:00,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2026-03-01 00:45:00,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2026-03-01 01:00:00,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0



Information about df3:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 9611 entries, 2026-03-01 00:00:00 to 2026-06-11 00:45:00
Data columns (total 87 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   B1_AUTO_POS          9611 non-null   float64
 1   B1_DIV_VLV_CLSD      9611 non-null   float64
 2   B1_DIV_VLV_OPN       9611 non-null   float64
 3   B1_EN                9611 non-null   float64
 4   B1_GAS_BSTR_STS      9611 non-null   float64
 5   B1_HND_POS           9611 non-null   float64
 6   B1_HTCO              9611 non-null   float64
 7   B1_LKOUT             9611 non-null   float64
 8   B1_RUN               9611 non-null   float64
 9   B2_AUTO_POS          9611 non-null   float64
 10  B2_DIV_VLV_CLSD      9611 non-null   float64
 11  B2_DIV_VLV_OPN       9611 non-null   float64
 12  B2_EN                9611 non-null   float64
 13  B2_GAS_BSTR_STS      9611 non-null   float64
 14  B2_HND_POS           9611 no

Becuase both `df1` and `df3` have datetime indices, we can merge them into a new DataFrame, `df4`, using an outer join to include all available timestamps from both datasets.

In [ ]:
# Store the original columns of df1 to filter df4 later
original_df1_columns = df1.columns.tolist()

# Combine df1 and df3 using combine_first to handle overlapping columns
# This method fills NaN values in df1 with corresponding values from df3.
# For columns present in both, df1's non-NaN values are prioritized.
# Columns only in df3 will be added temporarily at this stage.
df4 = df1.combine_first(df3)

# Filter df4 to keep only the columns that were originally in df1
# And ensure the column order is the same as the original df1
df4 = df4[original_df1_columns]

print("Combined DataFrame df4 created successfully, keeping only df1's original columns and order:")
display(df4.head())
display(df4.tail())
print("\nInformation about df4:")
df4.info()

You want to save `df4` to a new, separate Google Sheet file. I'll create a new spreadsheet in your Google Drive and write `df4` to its first tab. You will find the new sheet in your Google Drive's root folder.

In [ ]:
from gspread_dataframe import set_with_dataframe

try:
    # Define the name for the new spreadsheet file
    new_spreadsheet_name = 'Combined_Data_df4'

    # Create a new spreadsheet in Google Drive
    # By default, this creates the sheet in your Google Drive's root folder.
    new_sh = gc.create(new_spreadsheet_name)

    # Share the newly created spreadsheet to make it accessible (optional, but good practice)
    # You might need to adjust permissions based on your needs
    # For example, to make it viewable by anyone with the link:
    # new_sh.share(None, perm_type='anyone', role='reader')

    # Get the first worksheet of the newly created spreadsheet
    worksheet_df4 = new_sh.get_worksheet(0) # Index 0 is usually the first default sheet

    # Write df4 to the new worksheet, including the DatetimeIndex as a column
    set_with_dataframe(worksheet_df4, df4.reset_index())

    print(f"Successfully created new spreadsheet '{new_spreadsheet_name}' and saved df4 to it.")
    print(f"You can access the new spreadsheet here: {new_sh.url}")
except Exception as e:
    print(f"Error creating new Google Sheet and saving df4: {e}")
    print("Please ensure you have sufficient Google Drive permissions and that the authentication is valid.")

# Graphs to understand BMS Parameters

In [ ]:
import plotly.express as px

# Select the columns to plot
columns_to_plot = ['OUT_AIR_TMP', 'STH_VT_FLW_TMP', 'STH_VT_RTN_TMP', 'STH_VT_VLV_POS', 'STH_VT_PMPS_FLW_STS']

# Create the interactive plot using Plotly Express
fig = px.line(
    df3[columns_to_plot],
    x=df3.index, # Use the DatetimeIndex as the x-axis
    y=columns_to_plot,
    # title='Time Series of OUT_AIR_TMP, STH_VT_FLW_TMP, and STH_VT_RTN_TMP',
    # labels={'index': 'Date and Time', 'value': 'Value'}
)

# Update layout for better readability and interaction
fig.update_xaxes(
    rangeslider_visible=True
)

fig.show()